In [1]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
import pdfplumber


CURRENT = Path.cwd()

if CURRENT.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT.parent
else:
    PROJECT_ROOT = CURRENT


PDF_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "daily"
)

OUTPUT_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


PDF_NAME = "IDSP-Daily-Report-01.09.2026.pdf"

PDF_PATH = PDF_FOLDER / PDF_NAME


if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}"
    )


print("Using PDF:", PDF_PATH.name)

Using PDF: IDSP-Daily-Report-01.09.2026.pdf


In [2]:
def clean(value):
    if value is None:
        return ""

    text = str(value)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def to_state_count(value):
    text = clean(value)

    text = text.replace(",", "")

    if text == "-":
        return 0

    if text == "":
        return pd.NA

    if text.isdigit():
        return int(text)

    raise ValueError(
        f"Invalid state count: {value!r}"
    )

In [3]:
TABLE_SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "intersection_tolerance": 5,
    "snap_tolerance": 3,
    "join_tolerance": 3,
}


with pdfplumber.open(PDF_PATH) as pdf:
    if len(pdf.pages) < 3:
        raise ValueError(
            "The PDF does not contain page 3."
        )

    page_3 = pdf.pages[2]

    tables = page_3.extract_tables(
        TABLE_SETTINGS
    )


if not tables:
    raise ValueError(
        "No table was found on page 3."
    )


state_table = max(
    tables,
    key=len
)


row_count = len(state_table)

column_count = max(
    len(row) for row in state_table
)


print("Rows:", row_count)
print("Columns:", column_count)

Rows: 32
Columns: 15


In [4]:
header_text = " ".join(
    clean(cell)
    for row in state_table[:3]
    for cell in row
)


date_match = re.search(
    r"\b\d{2}-\d{2}-\d{2}\b",
    header_text
)


if date_match is None:
    raise ValueError(
        "The report date was not found."
    )


report_date = datetime.strptime(
    date_match.group(),
    "%d-%m-%y"
)


report_date = (
    report_date
    .date()
    .isoformat()
)


print("Report date:", report_date)

Report date: 2026-09-01


In [5]:
STATE_VALUE_COLUMNS = [
    "daily_suspected_cases",
    "daily_suspected_deaths",
    "daily_confirmed",
    "daily_deaths",

    "month_suspected_cases",
    "month_suspected_deaths",
    "month_confirmed",
    "month_deaths",

    "cumulative_suspected_cases",
    "cumulative_suspected_deaths",
    "cumulative_confirmed",
    "cumulative_deaths",
]


EXPECTED_COLUMN_COUNT = 15


if column_count != EXPECTED_COLUMN_COUNT:
    raise ValueError(
        "Unknown page-3 layout.\n"
        f"Expected columns: {EXPECTED_COLUMN_COUNT}\n"
        f"Detected columns: {column_count}"
    )


if len(state_table) < 3:
    raise ValueError(
        "The state table does not have enough header rows."
    )


main_header = state_table[1]


expected_headers = {
    0: "SL.NO",
    1: "DISEASE",
    3: "DAILY",
    7: "PRESENT MONTH",
    11: "CUMULATIVE",
}


header_errors = []


for position, expected_name in expected_headers.items():
    actual_name = clean(
        main_header[position]
    ).upper()

    if expected_name not in actual_name:
        header_errors.append({
            "position": position,
            "expected": expected_name,
            "detected": actual_name,
        })


if header_errors:
    raise ValueError(
        "Page-3 headers changed:\n"
        f"{header_errors}"
    )


print("PASS: Page 3 matches state_v1.")

PASS: Page 3 matches state_v1.


In [6]:
records = []

current_serial = None
current_disease = None


for row in state_table[3:]:

    if len(row) != EXPECTED_COLUMN_COUNT:
        continue

    serial_text = clean(row[0])

    disease_text = clean(row[1])

    subtype_text = clean(row[2])


    if serial_text.isdigit():
        current_serial = int(serial_text)
        current_disease = disease_text

    elif (
        serial_text == ""
        and disease_text == ""
        and subtype_text in {"Ind.", "Imp."}
        and current_disease is not None
    ):
        pass

    else:
        continue


    record = {
        "serial_number": current_serial,
        "disease_raw": current_disease,
        "subtype_raw": subtype_text,
    }


    values = row[3:15]


    for column, value in zip(
        STATE_VALUE_COLUMNS,
        values
    ):
        record[column] = to_state_count(
            value
        )


    records.append(record)


state_df = pd.DataFrame(records)


print("Extracted state rows:", len(state_df))

state_df[
    [
        "serial_number",
        "disease_raw",
        "subtype_raw",
        "daily_confirmed",
    ]
]

Extracted state rows: 26


,serial_number,disease_raw,subtype_raw,daily_confirmed
0,1,Fever,,12480
1,2,Malaria,Ind.,0
2,2,Malaria,Imp.,5
3,3,Dengue Fever,,79
4,4,Chikungunya,,1
5,5,Con JE,,0
6,6,AES,,0
7,7,Leptospirosis,,18
8,8,Hepatitis-A,,17
9,9,Hepatitis-E,,0


In [7]:
DISEASE_ALIASES = {
    "Dengue Fever": "Dengue",
    "Con JE": "Japanese Encephalitis",
    "AES": "Acute Encephalitis Syndrome",
    "Hepatitis-A": "Hepatitis A",
    "Hepatitis-E": "Hepatitis E",
    "ADD": "Acute Diarrhoeal Disease",
    "Chicken Pox": "Chickenpox",
    "M Pox": "Mpox",
    "Amebic Meningoencephalitis":
        "Amoebic Meningoencephalitis",
}


SUBTYPE_ALIASES = {
    "Ind.": "indigenous",
    "Imp.": "imported",
}


def normalize_disease(disease_name):
    return DISEASE_ALIASES.get(
        disease_name,
        disease_name
    )


def normalize_subtype(subtype_name):
    if subtype_name == "":
        return None

    return SUBTYPE_ALIASES.get(
        subtype_name,
        subtype_name
    )


state_df["disease"] = (
    state_df["disease_raw"]
    .map(normalize_disease)
)


state_df["subtype"] = (
    state_df["subtype_raw"]
    .map(normalize_subtype)
)


for column in STATE_VALUE_COLUMNS:
    state_df[column] = (
        state_df[column]
        .astype("Int64")
    )


state_df[
    [
        "disease_raw",
        "disease",
        "subtype",
        "daily_confirmed",
    ]
]

,disease_raw,disease,subtype,daily_confirmed
0,Fever,Fever,None,12480
1,Malaria,Malaria,indigenous,0
2,Malaria,Malaria,imported,5
3,Dengue Fever,Dengue,None,79
4,Chikungunya,Chikungunya,None,1
5,Con JE,Japanese Encephalitis,None,0
6,AES,Acute Encephalitis Syndrome,None,0
7,Leptospirosis,Leptospirosis,None,18
8,Hepatitis-A,Hepatitis A,None,17
9,Hepatitis-E,Hepatitis E,None,0


In [8]:
if state_df.empty:
    raise ValueError(
        "No disease rows were extracted."
    )


if state_df["disease_raw"].eq("").any():
    raise ValueError(
        "A disease row has no disease name."
    )


duplicate_keys = state_df.duplicated(
    subset=[
        "disease",
        "subtype",
    ],
    keep=False
)


if duplicate_keys.any():
    duplicates = state_df.loc[
        duplicate_keys,
        [
            "disease",
            "subtype",
        ]
    ]

    raise ValueError(
        "Duplicate disease/subtype rows found:\n"
        + duplicates.to_string(index=False)
    )


print(
    "PASS: Extracted",
    len(state_df),
    "state rows."
)

print(
    "Unique diseases:",
    state_df["disease"].nunique()
)

PASS: Extracted 26 state rows.
Unique diseases: 25


In [9]:
state_df.insert(
    0,
    "report_date",
    report_date
)


state_df.insert(
    1,
    "period_type",
    "daily"
)


state_df.insert(
    2,
    "source_filename",
    PDF_PATH.name
)


state_df.insert(
    3,
    "schema_version",
    "state_v1"
)


state_df.insert(
    4,
    "source_page",
    3
)

In [10]:
positive_daily_cases = state_df[
    state_df["daily_confirmed"]
    .fillna(0)
    .gt(0)
].copy()


positive_daily_cases = (
    positive_daily_cases
    .sort_values(
        by="daily_confirmed",
        ascending=False
    )
)


positive_daily_cases[
    [
        "disease",
        "subtype",
        "daily_suspected_cases",
        "daily_confirmed",
        "daily_deaths",
    ]
]

,disease,subtype,daily_suspected_cases,daily_confirmed,daily_deaths
0,Fever,None,0,12480,0
12,Acute Diarrhoeal Disease,None,0,1793,0
15,Chickenpox,None,0,83,0
16,Influenza,None,0,82,1
3,Dengue,None,174,79,1
7,Leptospirosis,None,11,18,2
8,Hepatitis A,None,41,17,1
14,Mumps,None,0,10,0
2,Malaria,imported,0,5,0
17,Scrub Typhus,None,0,2,0


In [11]:
nipah_result = state_df[
    state_df["disease"]
    .str.casefold()
    .eq("nipah")
]


nipah_result[
    [
        "report_date",
        "disease",
        "daily_confirmed",
        "cumulative_confirmed",
        "cumulative_deaths",
    ]
]

,report_date,disease,daily_confirmed,cumulative_confirmed,cumulative_deaths
21,2026-09-01,Nipah,0,1,0


In [12]:
state_output_path = (
    OUTPUT_FOLDER
    / f"state_analysis_{report_date}.csv"
)


state_df.to_csv(
    state_output_path,
    index=False
)


print("State analysis saved:")
print(state_output_path)

State analysis saved:
C:\Users\vinee\Downloads\rag_chatbot_kerala\data\processed\state_analysis_2026-09-01.csv


In [13]:
print("Notebook 02 completed successfully.")

print("Report date:", report_date)

print(
    "State rows extracted:",
    len(state_df)
)

print(
    "Unique diseases:",
    state_df["disease"].nunique()
)

print(
    "Daily diseases with confirmed cases:",
    len(positive_daily_cases)
)

print("Schema version: state_v1")

Notebook 02 completed successfully.
Report date: 2026-09-01
State rows extracted: 26
Unique diseases: 25
Daily diseases with confirmed cases: 13
Schema version: state_v1
